Import

In [65]:
import pandas as pd

pd.set_option(
    "display.max_columns",
    None
)

Drug Lookup

In [67]:
def drug_lookup(
    search_value
):

    search_value = (
        str(search_value)
        .upper()
        .strip()
    )

    return drug_master_enriched_df[

        drug_master_enriched_df[
            "brand_name"
        ]
        .fillna("")
        .str.upper()
        .str.contains(
            search_value,
            na=False
        )

        |

        drug_master_enriched_df[
            "generic_name"
        ]
        .fillna("")
        .str.upper()
        .str.contains(
            search_value,
            na=False
        )

        |

        drug_master_enriched_df[
            "product_ndc"
        ]
        .fillna("")
        .astype(str)
        .str.contains(
            search_value,
            na=False
        )

        |

        drug_master_enriched_df[
            "package_ndc"
        ]
        .fillna("")
        .astype(str)
        .str.contains(
            search_value,
            na=False
        )

    ]

Drug Profile

In [68]:
def drug_profile(
    search_value
):

    results = drug_lookup(
        search_value
    )

    if len(results) == 0:

        print(
            "No matches found."
        )

        return

    return results

Attributes

In [88]:
def ask_drug(
    search_value
):

    results = drug_lookup(
        search_value
    )

    if len(results) == 0:

        return "No matches found."

    drug = (
        results
        .iloc[0]
        .to_dict()
    )

    response = []

    response.append(
        f"Brand Name: {drug['brand_name']}"
    )

    response.append(
        f"Generic Name: {drug['generic_name']}"
    )

    response.append(
        f"Ingredient: {drug['ingredient_name']}"
    )

    response.append(
        f"Manufacturer: {drug['manufacturer']}"
    )

    response.append(
        f"Route: {drug['route']}"
    )

    response.append(
        f"Dosage Form: {drug['dosage_form']}"
    )

    response.append(
        f"Marketing Category: {drug['marketing_category']}"
    )

    response.append(
        f"Application Number: {drug['application_number']}"
    )

    response.append(
        f"Single Dose Vial: {drug['single_dose_vial']}"
    )

    response.append(
        f"Multi Dose Vial: {drug['multi_dose_vial']}"
    )

    response.append(
        f"Preservative Free: {drug['preservative_free']}"
    )

    response.append(
        f"Refrigerated: {drug['refrigerated']}"
    )

    response.append(
        f"Do Not Freeze: {drug['do_not_freeze']}"
    )

    response.append(
        f"Do Not Shake: {drug['do_not_shake']}"
    )

    return "\n".join(
        response
    )

In [70]:
dailymed_attributes_df = pd.DataFrame(

    columns=[

        "brand_name",

        "single_dose_vial",

        "multi_dose_vial",

        "preservative_free",

        "refrigerated",

        "do_not_freeze",

        "do_not_shake"
    ]
)


In [72]:
unique_brands = (

    drug_master_enriched_df[
        "brand_name"
    ]

    .dropna()

    .drop_duplicates()

    .sort_values()

    .reset_index(
        drop=True
    )
)

print(
    f"Unique Brands: {len(unique_brands):,}"
)

Unique Brands: 4,953


DailyMed

In [73]:
import requests

def get_dailymed_setid(
    brand_name
):

    response = requests.get(

        "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json",

        params={
            "drug_name":
                brand_name
        },

        timeout=30
    )

    data = response.json()

    records = data.get(
        "data",
        []
    )

    if len(records) == 0:

        return None

    return records[0][
        "setid"
    ]

DailyMed Attributes

In [74]:
from bs4 import BeautifulSoup

def extract_dailymed_attributes(
    brand_name
):

    setid = get_dailymed_setid(
        brand_name
    )

    if setid is None:

        return None

    response = requests.get(

        f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{setid}.xml",

        timeout=30
    )

    soup = BeautifulSoup(
        response.text,
        "xml"
    )

    full_text = soup.get_text(
        separator=" ",
        strip=True
    ).lower()

    return {

        "brand_name":
            brand_name,

        "setid":
            setid,

        "single_dose_vial":
            "single-dose vial"
            in full_text,

        "multi_dose_vial":
            "multi-dose vial"
            in full_text,

        "preservative_free":
            "does not contain a preservative"
            in full_text,

        "refrigerated":
            "2°c to 8°c"
            in full_text,

        "do_not_freeze":
            "do not freeze"
            in full_text,

        "do_not_shake":
            "do not shake"
            in full_text
    }

Build Repository

In [75]:
def build_dailymed_attributes_repository(
    brand_names
):

    rows = []

    total = len(
        brand_names
    )

    for index, brand_name in enumerate(
        brand_names,
        start=1
    ):

        print(
            f"{index:,} / {total:,}"
        )

        try:

            attributes = (
                extract_dailymed_attributes(
                    brand_name
                )
            )

            if attributes:

                rows.append(
                    attributes
                )

        except Exception as e:

            print(
                f"ERROR: {brand_name}"
            )

    return pd.DataFrame(
        rows
    )

In [76]:
brand_names = (

    drug_master_enriched_df[
        "brand_name"
    ]

    .dropna()

    .drop_duplicates()

    .tolist()
)

In [78]:
dailymed_attributes_df = (
    build_dailymed_attributes_repository(
        brand_names[:100]
    )
)

dailymed_attributes_df.head()

1 / 100
2 / 100
3 / 100
4 / 100
5 / 100
6 / 100
7 / 100
8 / 100
9 / 100
10 / 100
11 / 100
12 / 100
13 / 100
14 / 100
15 / 100
16 / 100
17 / 100
18 / 100
19 / 100
20 / 100
21 / 100
22 / 100
23 / 100
24 / 100
25 / 100
26 / 100
27 / 100
28 / 100
29 / 100
30 / 100
31 / 100
32 / 100
33 / 100
34 / 100
35 / 100
36 / 100
37 / 100
38 / 100
39 / 100
40 / 100
41 / 100
42 / 100
43 / 100
44 / 100
45 / 100
46 / 100
47 / 100
48 / 100
49 / 100
50 / 100
51 / 100
52 / 100
53 / 100
54 / 100
55 / 100
56 / 100
57 / 100
58 / 100
59 / 100
60 / 100
61 / 100
62 / 100
63 / 100
64 / 100
65 / 100
66 / 100
67 / 100
68 / 100
69 / 100
70 / 100
71 / 100
72 / 100
73 / 100
74 / 100
75 / 100
76 / 100
77 / 100
78 / 100
79 / 100
80 / 100
81 / 100
82 / 100
83 / 100
84 / 100
85 / 100
86 / 100
87 / 100
88 / 100
89 / 100
90 / 100
91 / 100
92 / 100
93 / 100
94 / 100
95 / 100
96 / 100
97 / 100
98 / 100
99 / 100
100 / 100


,brand_name,setid,single_dose_vial,multi_dose_vial,preservative_free,refrigerated,do_not_freeze,do_not_shake
0,TRIDERMA EVERYDAY BRUISE RELIEF,5100f2d0-cfa2-4136-bda4-860de81707c8,False,False,False,False,False,False
1,Oxy Advanced Care Rapid Spot Treatment,e613e8a6-ffed-49bc-bdc0-87606c51ffa3,False,False,False,False,False,False
2,medicated pads,2d020e79-2bcf-ba8f-e063-6394a90a6fda,False,False,False,False,False,False
3,Celecoxib,d94849a7-1831-4319-982c-c6b20dc42038,False,False,False,True,False,False
4,TRIAMCINOLONE ACETONIDE,5f3aaf3a-cc66-4d4f-b9b7-2c50c325d477,True,False,False,False,False,False


In [77]:
dailymed_attributes_df.to_parquet(

    "dailymed_repository/dailymed_attributes.parquet",

    index=False
)

print(
    f"Saved {len(dailymed_attributes_df):,} rows."
)

Saved 0 rows.


In [25]:
saved_df = pd.read_parquet(

    "dailymed_repository/dailymed_attributes.parquet"
)

saved_df.shape

(100, 8)

Backup Cell

In [26]:
dailymed_attributes_df.to_parquet(

    "dailymed_repository/dailymed_attributes_backup.parquet",

    index=False
)

Start Here

In [99]:
processed_brands = set(
    dailymed_attributes_df[
        "brand_name"
    ]
    .dropna()
    .unique()
)

remaining_brands = [

    brand

    for brand in brand_names

    if brand not in processed_brands
]

print(
    f"Processed: {len(processed_brands):,}"
)

print(
    f"Remaining: {len(remaining_brands):,}"
)

Processed: 200
Remaining: 4,753


In [100]:
batch_df = (
    build_dailymed_attributes_repository(
        remaining_brands[:800]
    )
)

1 / 800
2 / 800
3 / 800
4 / 800
5 / 800
6 / 800
7 / 800
8 / 800
9 / 800
10 / 800
11 / 800
12 / 800
13 / 800
14 / 800
15 / 800
16 / 800
17 / 800
18 / 800
19 / 800
20 / 800
21 / 800
22 / 800
23 / 800
24 / 800
25 / 800
26 / 800
27 / 800
28 / 800
29 / 800
30 / 800
31 / 800
32 / 800
33 / 800
34 / 800
35 / 800
36 / 800
37 / 800
38 / 800
39 / 800
40 / 800
41 / 800
42 / 800
43 / 800
44 / 800
45 / 800
46 / 800
47 / 800
48 / 800
49 / 800
50 / 800
51 / 800
52 / 800
53 / 800
54 / 800
55 / 800
56 / 800
57 / 800
58 / 800
59 / 800
60 / 800
61 / 800
62 / 800
63 / 800
64 / 800
65 / 800
66 / 800
67 / 800
68 / 800
69 / 800
70 / 800
71 / 800
72 / 800
73 / 800
74 / 800
75 / 800
76 / 800
77 / 800
78 / 800
79 / 800
80 / 800
81 / 800
82 / 800
83 / 800
84 / 800
85 / 800
86 / 800
87 / 800
88 / 800
89 / 800
90 / 800
91 / 800
92 / 800
93 / 800
94 / 800
95 / 800
96 / 800
97 / 800
98 / 800
99 / 800
100 / 800
101 / 800
102 / 800
103 / 800
104 / 800
105 / 800
106 / 800
107 / 800
108 / 800
109 / 800
110 / 800
111 / 80

In [111]:
dailymed_attributes_df = pd.concat(

    [
        dailymed_attributes_df,
        batch_df
    ],

    ignore_index=True
)

dailymed_attributes_df = (
    dailymed_attributes_df
    .drop_duplicates(
        subset=["brand_name"]
    )
)

dailymed_attributes_df.to_parquet(

    "dailymed_repository/dailymed_attributes.parquet",

    index=False
)

print(
    f"Saved {len(dailymed_attributes_df):,} brands"
)

Saved 998 brands


In [112]:
dailymed_attributes_df[
    [
        "single_dose_vial",
        "multi_dose_vial",
        "preservative_free",
        "refrigerated",
        "do_not_freeze",
        "do_not_shake"
    ]
].sum()

single_dose_vial     29
multi_dose_vial       5
preservative_free     2
refrigerated         39
do_not_freeze        46
do_not_shake         13
dtype: int64

In [113]:
dailymed_attributes_df = pd.read_parquet(
    "dailymed_repository/dailymed_attributes.parquet"
)

processed_brands = set(
    dailymed_attributes_df[
        "brand_name"
    ]
)

remaining_brands = [

    brand

    for brand in brand_names

    if brand not in processed_brands
]

print(
    f"Processed: {len(processed_brands):,}"
)

print(
    f"Remaining: {len(remaining_brands):,}"
)

Processed: 998
Remaining: 3,955


In [114]:
drug_master_enriched_df.to_parquet(

    "master_repository/drug_master_enriched.parquet",

    index=False
)